In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://abpdu.lbl.gov/"
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

records = []
current_category = "Uncategorized"

for element in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p"]):
    if element.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
        heading_text = element.get_text(strip=True)
        if heading_text:
            current_category = heading_text
    elif element.name == "p":
        paragraph_text = element.get_text(separator=" ", strip=True)
        if paragraph_text:
            records.append({
                "Data Source": url,
                "Category": current_category,
                "Content": paragraph_text
            })

df = pd.DataFrame(records, columns=["Data Source", "Category", "Content"])
df.to_csv("organized_abpdu_data.csv", index=False)
df.head()

In [ ]:
# send request with a Chrome-like User-Agent to avoid 403
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/115.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

records = []
current_category = "Uncategorized"

for element in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p"]):
    if element.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
        heading_text = element.get_text(strip=True)
        if heading_text:
            current_category = heading_text
    elif element.name == "p":
        paragraph_text = element.get_text(separator=" ", strip=True)
        if paragraph_text:
            records.append({
                "Data Source": url,
                "Category": current_category,
                "Content": paragraph_text
            })

df = pd.DataFrame(records, columns=["Data Source", "Category", "Content"])
df.to_csv("organized_abpdu_data.csv", index=False)
df.head()

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
import pandas as pd
import re
import os

# 1. Initialize variables in the correct sequence
video_id = "Wf798r6bNf0"
url = f"https://www.youtube.com/watch?v={video_id}"

try:
    # 2. Fetch the text using the correct library call
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    
    # 3. Process data snippets cleanly
    snippets = []
    for seg in transcript_list:
        text = seg.get("text", "")
        # Remove bracketed text like [Music] or [Laughter]
        text = re.sub(r"\[.*?\]", " ", text)
        # Collapse multiple spacing gaps
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            snippets.append(text)
    
    continuous_text = " ".join(snippets)
    
except Exception as e:
    continuous_text = f"Transcript unavailable: {e}"

# 4. Standardize the columns to match your database schema
df_transcript = pd.DataFrame([{
    "Data Source": url,
    "Category": f"YouTube Transcript ({video_id})",
    "Content": continuous_text
}], columns=["Data Source", "Category", "Content"])

# 5. Handle CSV file appending rules smoothly
csv_path = "organized_abpdu_data.csv"
if os.path.exists(csv_path):
    df_transcript.to_csv(csv_path, index=False, mode="a", header=False)
else:
    df_transcript.to_csv(csv_path, index=False)

# 6. Render the data row view
df_transcript.head()